# Character & Atmosphere Extraction — Showcase

This notebook is a thin runner over the `utils` package: it wires the pieces together and shows the output at each step. All of the actual logic (chunking, name detection, snippet gathering, LLM calls) lives in [`utils/`](utils).

Pipeline:
1. Chunk the book text
2. Detect character names (spaCy NER) and count mentions
3. Gather text snippets per character
4. Ask an LLM to extract a structured profile per character
5. Ask an LLM to extract the book's overall atmosphere

In [ ]:
from utils import (
    chunk_text,
    count_character_names,
    extract_book_atmosphere,
    extract_character_profiles,
    get_character_snippets,
    get_client,
    load_nlp,
    read_text,
    write_json,
)

BOOK_PATH = 'Data/Dracula - Bram Stoker.txt'
CHUNK_SIZE = 2000
NAME_THRESHOLD = 5  # only keep characters mentioned at least this many times

## 1. Chunk the book

In [ ]:
text = read_text(BOOK_PATH)
chunks = chunk_text(text, CHUNK_SIZE)

print(f'Chunks: {len(chunks)}')
chunks[10]

## 2. Detect and count character names

Runs spaCy NER over the full text and counts `PERSON` mentions.

In [ ]:
nlp = load_nlp()
name_counts = count_character_names(text, nlp, threshold=NAME_THRESHOLD)

write_json(name_counts, 'Data/character_glossary.json')
name_counts

## 3. Gather snippets per character

For each detected character, collect every chunk that mentions them.

In [ ]:
character_snippets = get_character_snippets(chunks, name_counts.keys())
character_snippets['count dracula']

## 4. Extract character profiles with the LLM

For each character, a few of their snippets are sent to the LLM as context so it can extract structured attributes (appearance, personality, vibe, ...).

In [ ]:
client = get_client()
character_profiles = extract_character_profiles(client, character_snippets)

write_json(character_profiles, 'Data/extracted_character_profiles.json')
character_profiles

## 5. Extract the book's atmosphere

Samples chunks from the beginning, middle, and end of the book and asks the LLM to synthesize the overall visual mood and tone.

In [ ]:
book_atmosphere = extract_book_atmosphere(client, chunks)

write_json(book_atmosphere, 'Data/book_atmosphere.json')
book_atmosphere